# Conformal prediction

Paper-facing tables for the 90% conformal prediction experiments. The analysis uses the linear probe, excludes TENX143 and Hibou-L, retains only LEMON ViT-S/8 under the label LEMON, and labels the CellBridge all/adapted representation simply as CellBridge.

The first table compares representations under branch split, the proposed conformal construction. The second fixes the CellBridge representation and compares the proposed branch split with global split, global graph, and class split. HEST (ID) and all OOD benchmarks appear in the same table.

In [1]:
import os
from pathlib import Path

import pandas as pd
from IPython.display import display

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 100)

WORKSPACE = Path(os.environ.get(
    "CELLBRIDGE_WORKSPACE",
    "/lustre/scratch126/cellgen/lotfollahi/so25",
))
RESULTS_ROOT = Path(os.environ.get(
    "CELLBRIDGE_INFERENCE_RESULTS",
    WORKSPACE / "outputs/reproducibility/inference",
))

PROBE = "linear"
EXCLUDED_HEST_SAMPLES = {"TENX143"}
METHOD_COMPARISON_CP = "branch_split"

METHOD_GROUPS = [
    ("Patch-level FMs", ["UNI2-h", "Virchow2"]),
    ("Cell-level FMs", ["VOLTA", "LEMON"]),
    ("Multi-modal FMs", ["PAST", "SpatialWhisperer"]),
    ("Ours", ["CellBridge"]),
]
METHOD_ORDER = [
    method
    for _, methods in METHOD_GROUPS
    for method in methods
]

ABLATION_ORDER = ["split", "graph", "class_split", "branch_split"]
CONFORMAL_LABELS = {
    "split": "Global split",
    "graph": "Global graph",
    "class_split": "Class split",
    "branch_split": "Branch split (ours)",
}

BENCHMARK_ORDER = [
    "HEST",
    "Lizard",
    "NuCLS",
    "PanNuke",
    "PhenoCell (all)",
    "PhenoCell (immune)",
]
BENCHMARK_LABELS = {
    "HEST": "HEST (ID)",
    "Lizard": "Lizard (OOD)",
    "NuCLS": "NuCLS (OOD)",
    "PanNuke": "PanNuke (OOD)",
    "PhenoCell (all)": "PhenoCell all (OOD)",
    "PhenoCell (immune)": "PhenoCell immune (OOD)",
}

TABLE_METRICS = [
    ("marginal_coverage", "Marginal coverage ↑"),
    ("mean_set_size", "Mean set size ↓"),
    ("singleton_rate", "Singleton rate ↑"),
]
LATEX_METRIC_LABELS = {
    "marginal_coverage": r"Marginal coverage $\uparrow$",
    "mean_set_size": r"Mean set size $\downarrow$",
    "singleton_rate": r"Singleton rate $\uparrow$",
}

ROW_COLOUR = "gray!15"
OURS_BACKGROUND = "#E6E6E6"

## Load and aggregate the conformal metrics

HEST entries are means across the 14 retained tissue sections; each OOD entry comes from its fixed held-out test split. Marginal coverage and singleton rate are reported as percentages. Mean set size is the mean number of labels retained per prediction set.

In [2]:
BASELINE_LABELS = {
    "uni": "UNI2-h",
    "virchow2": "Virchow2",
    "lemon": "LEMON",
    "past": "PAST",
    "volta": "VOLTA",
    "spatialwhisperer": "SpatialWhisperer",
}


def paper_method(method, version, arm):
    if method == "cellbridge" or arm.startswith("lora_"):
        size = version.removeprefix("cells_")
        size = size.upper() if size != "all" else "all"
        representation = arm.removeprefix("lora_")
        if size == "all" and representation == "adapted":
            return "CellBridge"
        return "CellBridge {} ({})".format(size, representation)
    return BASELINE_LABELS.get(method, method)


def benchmark_label(dataset, task):
    if dataset == "phenocell":
        return "PhenoCell ({})".format(task)
    return {
        "hest": "HEST",
        "lizard": "Lizard",
        "nucls": "NuCLS",
        "pannuke": "PanNuke",
    }[dataset]


def load_conformal_metrics(root):
    paths = list(root.glob(
        "*/*/*/evaluation/*/*/*/conformal_prediction/metrics.csv"
    ))
    paths += list(root.glob(
        "*/*/hest/*/evaluation/*/*/*/conformal_prediction/metrics.csv"
    ))
    tables = []
    required = {
        "method",
        "arm",
        "task",
        "probe",
        "conformal_method",
        "target_coverage",
        "marginal_coverage",
        "mean_prediction_set_size",
        "singleton_rate",
    }
    for path in sorted(paths):
        parts = path.relative_to(root).parts
        method, version, dataset = parts[:3]
        if dataset == "hest":
            sample_id = parts[3]
            arm, task, probe = parts[5:8]
        else:
            sample_id = pd.NA
            arm, task, probe = parts[4:7]

        table = pd.read_csv(path)
        missing = required.difference(table.columns)
        if missing:
            raise ValueError(
                "{} is missing columns: {}".format(path, sorted(missing))
            )
        table["version"] = version
        table["sample_id"] = sample_id
        table["paper_method"] = paper_method(method, version, arm)
        table["benchmark"] = benchmark_label(dataset, task)
        table["result_dir"] = str(path.parent)
        tables.append(table)

    if not tables:
        raise FileNotFoundError(
            "No conformal metrics found below {}".format(root)
        )

    table = pd.concat(tables, ignore_index=True)
    table = table.loc[table["method"] != "hibou_l"]
    table = table.loc[
        ~((table["method"] == "lemon") & (table["version"] != "vits8"))
    ]
    table = table.loc[
        ~table["sample_id"].astype(str).isin(EXCLUDED_HEST_SAMPLES)
    ]
    return table.reset_index(drop=True)


def aggregate_points(table):
    return (
        table.groupby(
            ["paper_method", "benchmark", "probe", "conformal_method"],
            as_index=False,
        )
        .agg(
            marginal_coverage=("marginal_coverage", "mean"),
            mean_set_size=("mean_prediction_set_size", "mean"),
            singleton_rate=("singleton_rate", "mean"),
            target_coverage=("target_coverage", "mean"),
            n_hest_sections=("sample_id", "count"),
        )
    )


conformal = load_conformal_metrics(RESULTS_ROOT)
points = aggregate_points(conformal)

target_coverages = conformal["target_coverage"].dropna().unique()
if len(target_coverages) != 1:
    raise ValueError(
        "Expected one nominal coverage target, found {}".format(
            sorted(target_coverages)
        )
    )
NOMINAL_COVERAGE = float(target_coverages[0])

print(
    "Loaded {:,} conformal rows at {:.0%} nominal coverage; "
    "probe = {}; excluded HEST = {}".format(
        len(conformal),
        NOMINAL_COVERAGE,
        PROBE,
        sorted(EXCLUDED_HEST_SAMPLES),
    )
)

Loaded 4,256 conformal rows at 90% nominal coverage; probe = linear; excluded HEST = ['TENX143']


## Representation comparison under branch split

This table compares all baseline representations with CellBridge using branch split and the linear probe. Datasets are rows and each representation has three metric columns. The gray columns are our proposed representation and conformal construction. HEST is the ID benchmark; the other five rows are OOD benchmarks.


In [3]:
def _format_metric(value, metric):
    if pd.isna(value):
        return "--"
    if metric in {"marginal_coverage", "singleton_rate"}:
        return "{:.1f}".format(100 * value)
    return "{:.2f}".format(value)


def build_transposed_table(
    rows,
    column_key,
    column_order,
    column_labels=None,
):
    duplicate = rows.duplicated([column_key, "benchmark"], keep=False)
    if duplicate.any():
        raise ValueError(
            "Duplicate table entries:\n{}".format(
                rows.loc[
                    duplicate,
                    [column_key, "benchmark"],
                ].to_string(index=False)
            )
        )

    observed = set(
        rows[[column_key, "benchmark"]].itertuples(index=False, name=None)
    )
    missing = [
        (column, benchmark)
        for column in column_order
        for benchmark in BENCHMARK_ORDER
        if (column, benchmark) not in observed
    ]
    if missing:
        raise ValueError("Missing table entries: {}".format(missing))

    metric_keys = [metric for metric, _ in TABLE_METRICS]
    raw = rows.pivot(
        index="benchmark",
        columns=column_key,
        values=metric_keys,
    )
    labels = column_labels or {}
    formatted = pd.DataFrame(
        index=[BENCHMARK_LABELS[benchmark] for benchmark in BENCHMARK_ORDER]
    )
    display_columns = []
    for column in column_order:
        column_label = labels.get(column, column)
        for metric, metric_label in TABLE_METRICS:
            display_column = (column_label, metric_label)
            formatted[display_column] = [
                _format_metric(raw.loc[benchmark, (metric, column)], metric)
                for benchmark in BENCHMARK_ORDER
            ]
            display_columns.append(display_column)
    formatted.index.name = "Dataset"
    formatted.columns = pd.MultiIndex.from_tuples(
        display_columns,
        names=["Method", "Metric"],
    )
    return formatted


def display_table(table, shaded_columns, label_level):
    styler = table.style.set_properties(**{"text-align": "center"})
    shaded = [
        column
        for column in table.columns
        if column[label_level] in shaded_columns
    ]
    if shaded:
        styler = styler.set_properties(
            subset=pd.IndexSlice[:, shaded],
            **{"background-color": OURS_BACKGROUND},
        )
    display(styler)


def _header_segments(columns, level):
    segments = []
    start = 0
    for position in range(1, len(columns) + 1):
        if (
            position == len(columns)
            or columns[position][level] != columns[start][level]
        ):
            segments.append((columns[start][level], start, position - 1))
            start = position
    return segments


def emit_latex_table(
    table,
    caption,
    label,
    shaded_columns,
    label_level,
):
    shaded = set(shaded_columns)
    column_spec = ["l"]
    for column in table.columns:
        if column[label_level] in shaded:
            column_spec.append(
                r">{\columncolor{" + ROW_COLOUR + r"}}c"
            )
        else:
            column_spec.append("c")

    lines = [
        r"\begin{table*}[t]",
        r"\centering",
        r"\caption{" + caption + "}",
        r"\label{" + label + "}",
        r"\resizebox{\textwidth}{!}{%",
        r"\begin{tabular}{" + "".join(column_spec) + "}",
        r"\toprule",
    ]

    for level in range(table.columns.nlevels - 1):
        header_cells = []
        rules = []
        for heading, start, end in _header_segments(table.columns, level):
            is_shaded = all(
                table.columns[position][label_level] in shaded
                for position in range(start, end + 1)
            )
            content = r"\textbf{" + str(heading) + "}"
            if is_shaded:
                content = r"\cellcolor{" + ROW_COLOUR + "}" + content
            header_cells.append(
                r"\multicolumn{"
                + str(end - start + 1)
                + r"}{c}{"
                + content
                + "}"
            )
            rules.append(
                r"\cmidrule(lr){"
                + "{}-{}".format(start + 2, end + 2)
                + "}"
            )
        lines.append(" & " + " & ".join(header_cells) + r" \\")
        lines.append(" ".join(rules))

    latex_metric_labels = {
        display_label: LATEX_METRIC_LABELS[metric]
        for metric, display_label in TABLE_METRICS
    }
    metric_headers = [
        latex_metric_labels[column[-1]] for column in table.columns
    ]
    lines.append("Dataset & " + " & ".join(metric_headers) + r" \\")
    lines.append(r"\midrule")

    for dataset, row in table.iterrows():
        lines.append(
            str(dataset)
            + " & "
            + " & ".join(row.astype(str).tolist())
            + r" \\"
        )

    lines.extend([
        r"\bottomrule",
        r"\end{tabular}",
        r"}",
        r"\end{table*}",
    ])
    print("\n".join(lines))


main_rows = points.loc[
    (points["probe"] == PROBE)
    & (points["conformal_method"] == METHOD_COMPARISON_CP)
    & points["paper_method"].isin(METHOD_ORDER)
].copy()
main_table = build_transposed_table(
    main_rows,
    column_key="paper_method",
    column_order=METHOD_ORDER,
)
family_by_method = {
    method: family
    for family, methods in METHOD_GROUPS
    for method in methods
}
main_table.columns = pd.MultiIndex.from_tuples(
    [
        (family_by_method[method], method, metric)
        for method, metric in main_table.columns
    ],
    names=["Family", "Method", "Metric"],
)

main_shaded_columns = {"CellBridge"}
display_table(
    main_table,
    shaded_columns=main_shaded_columns,
    label_level=1,
)
emit_latex_table(
    main_table,
    caption=(
        "Representation comparison under branch split using linear probes. "
        "Marginal coverage and singleton rate are percentages; the nominal "
        "{:.0f}\\% coverage target should be considered together with mean "
        "set size and singleton rate. HEST is ID and all other benchmarks "
        "are OOD. HEST entries average 14 retained sections. Gray columns "
        "identify CellBridge."
    ).format(100 * NOMINAL_COVERAGE),
    label="tab:conformal-representation-comparison",
    shaded_columns=main_shaded_columns,
    label_level=1,
)


\begin{table*}[t]
\centering
\caption{Representation comparison under branch split using linear probes. Marginal coverage and singleton rate are percentages; the nominal 90\% coverage target should be considered together with mean set size and singleton rate. HEST is ID and all other benchmarks are OOD. HEST entries average 14 retained sections. Gray columns identify CellBridge.}
\label{tab:conformal-representation-comparison}
\resizebox{\textwidth}{!}{%
\begin{tabular}{lcccccccccccccccccc>{\columncolor{gray!15}}c>{\columncolor{gray!15}}c>{\columncolor{gray!15}}c}
\toprule
 & \multicolumn{6}{c}{\textbf{Patch-level FMs}} & \multicolumn{6}{c}{\textbf{Cell-level FMs}} & \multicolumn{6}{c}{\textbf{Multi-modal FMs}} & \multicolumn{3}{c}{\cellcolor{gray!15}\textbf{Ours}} \\
\cmidrule(lr){2-7} \cmidrule(lr){8-13} \cmidrule(lr){14-19} \cmidrule(lr){20-22}
 & \multicolumn{3}{c}{\textbf{UNI2-h}} & \multicolumn{3}{c}{\textbf{Virchow2}} & \multicolumn{3}{c}{\textbf{VOLTA}} & \multicolumn{3}{c}{\te

## CellBridge conformal-construction ablation

All columns use the same CellBridge representation and linear probe. Global split is the standard marginal conformal baseline, global graph adds graph-based risk control, class split calibrates each label separately, and branch split calibrates the proposed biological branches.

Class split is included because it directly tests whether fine-grained label conditioning is preferable to branch-level conditioning. It can be less stable for labels with limited calibration support, so it should be interpreted alongside mean set size and singleton rate. The gray columns are the proposed branch-split configuration.


In [4]:
ablation_rows = points.loc[
    (points["probe"] == PROBE)
    & (points["paper_method"] == "CellBridge")
    & points["conformal_method"].isin(ABLATION_ORDER)
].copy()
ablation_table = build_transposed_table(
    ablation_rows,
    column_key="conformal_method",
    column_order=ABLATION_ORDER,
    column_labels=CONFORMAL_LABELS,
)
ablation_table.columns.names = ["Configuration", "Metric"]

proposed_configuration = {"Branch split (ours)"}
display_table(
    ablation_table,
    shaded_columns=proposed_configuration,
    label_level=0,
)
emit_latex_table(
    ablation_table,
    caption=(
        "CellBridge conformal-construction ablation using linear probes. "
        "Marginal coverage and singleton rate are percentages; the nominal "
        "{:.0f}\\% coverage target should be considered together with mean "
        "set size and singleton rate. HEST is ID and all other benchmarks "
        "are OOD. Gray columns identify the proposed branch-split "
        "construction."
    ).format(100 * NOMINAL_COVERAGE),
    label="tab:conformal-construction-ablation",
    shaded_columns=proposed_configuration,
    label_level=0,
)


\begin{table*}[t]
\centering
\caption{CellBridge conformal-construction ablation using linear probes. Marginal coverage and singleton rate are percentages; the nominal 90\% coverage target should be considered together with mean set size and singleton rate. HEST is ID and all other benchmarks are OOD. Gray columns identify the proposed branch-split construction.}
\label{tab:conformal-construction-ablation}
\resizebox{\textwidth}{!}{%
\begin{tabular}{lccccccccc>{\columncolor{gray!15}}c>{\columncolor{gray!15}}c>{\columncolor{gray!15}}c}
\toprule
 & \multicolumn{3}{c}{\textbf{Global split}} & \multicolumn{3}{c}{\textbf{Global graph}} & \multicolumn{3}{c}{\textbf{Class split}} & \multicolumn{3}{c}{\cellcolor{gray!15}\textbf{Branch split (ours)}} \\
\cmidrule(lr){2-4} \cmidrule(lr){5-7} \cmidrule(lr){8-10} \cmidrule(lr){11-13}
Dataset & Marginal coverage $\uparrow$ & Mean set size $\downarrow$ & Singleton rate $\uparrow$ & Marginal coverage $\uparrow$ & Mean set size $\downarrow$ & Singleto

## Reading the tables

- Marginal coverage is a validity criterion. Values below the 90% target indicate undercoverage; values far above it are conservative rather than automatically better.
- Mean set size and singleton rate are complementary efficiency measures. Smaller sets and more singleton predictions are desirable only while coverage remains adequate.
- HEST is the ID benchmark and is averaged across the 14 retained tissue sections. Lizard, NuCLS, PanNuke, PhenoCell (all), and PhenoCell (immune) are OOD benchmarks.
- The main table shades the CellBridge columns because they combine our representation with the proposed branch split. The ablation shades only the branch-split columns so the proposed construction remains identifiable when the representation is fixed.
